# Aerial Fire Detection with Drone Imagery and Computer Vision

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Start here.** This notebook is the guided walkthrough for the whole project: what
we are building, how the system fits together, how the dataset was made, and where
each piece of runnable code lives.

It is the map. The three notebooks it points you to are the territory:

| # | Notebook | What it does |
|---|---|---|
| 1 | [`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb) | Trains the fire detector on the Roboflow dataset |
| 2 | [`Supervision_Image_Inferencing.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Image_Inferencing.ipynb) | Runs the trained model on a single image |
| 3 | [`Supervision_Video_Inferencing.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Video_Inferencing.ipynb) | Runs detection + ByteTrack tracking across a video |

Work through this notebook top to bottom; it will tell you when to jump into each one.

*This walkthrough follows the project described in Roboflow's write-up,
[Aerial Fire Detection with Drone Imagery and Computer Vision](https://blog.roboflow.com/aerial-fire-detection/),
by Timothy M. Full citation at the end. The code here has been brought up to date —
the published article targets 2023-era libraries that no longer run.*

## Why aerial fire detection

Wildfire detection has traditionally leaned on two things: ground-based sensors and
satellite imagery. Both have real limits. Ground equipment only sees where you have
already put it, and it is expensive to blanket a forest. Satellites cover enormous
areas but at a resolution and revisit cadence that can miss a fire during the window
when it is still small enough to control cheaply.

That window is the whole game. A fire caught in its first minutes is a very different
problem from the same fire caught an hour later.

A drone carrying a camera and a computer vision model sits in the gap between those
two approaches. It covers ground far faster than a person on foot, flies low enough
to see detail a satellite cannot, and can be sent out on a schedule across terrain
nobody could realistically patrol daily.

## What the system needs

Four things, two physical and two software:

| | Component | Role |
|---|---|---|
| 🛩️ | **Drone** | Carries the sensor over terrain that is slow or unsafe to patrol on foot |
| 📷 | **WiFi camera module** | Captures the imagery and streams it back to the ground |
| 🏷️ | **Roboflow account** | Hosts the dataset, extracts video frames, and provides the labelling tools |
| 📓 | **Google Colab** | Free GPU for training the model — this repo's notebooks all run there |

## How the system works end to end

Detection is only the middle of the story. The value comes from what happens on
either side of it — getting the camera over the right ground, and getting a human
response moving once something is found.

```
   ┌──────────────┐
   │  1. DRONE    │  Deployed over the survey area, covering ground
   │   DEPLOYED   │  quickly and reaching terrain that is hard to patrol
   └──────┬───────┘
          │  video / stills
          ▼
   ┌──────────────┐
   │ 2. REMOTE    │  Trained personnel fly and monitor the aircraft
   │  INSPECTION  │  from the ground station
   └──────┬───────┘
          │  frames
          ▼
   ┌──────────────┐
   │ 3. COMPUTER  │  ← THIS is what the three notebooks build
   │    VISION    │    YOLO26 scans every frame for flame and smoke
   └──────┬───────┘
          │  detections + track IDs
          ▼
   ┌──────────────┐
   │ 4. FIRE      │  A confirmed detection raises an immediate alert
   │  DETECTED    │
   └──────┬───────┘
          ▼
   ┌──────────────┐
   │ 5. CONTROL   │  Operators assess the alert and decide the response
   │   CENTRE     │
   └──────┬───────┘
          ▼
   ┌──────────────┐
   │ 6. RESPONSE  │  Crews are dispatched to the detected location
   │  TEAM SENT   │
   └──────────────┘
```

Steps 1, 2, 5 and 6 are operational — aircraft, people, procedure. **Step 3 is the
part this repository actually implements**, and step 4 falls out of it. The model
looks for the visual signatures of fire: visible flame, smoke plumes, and the
discolouration of burning ground.

The four build steps that get you there:

1. **Prepare a dataset** — collect the raw aerial footage
2. **Label it and generate a version** — draw the boxes, snapshot the result
3. **Train a model** — notebook 1
4. **Test the model** — notebooks 2 and 3

## Step 1 — Dataset preparation

The source is the **FLAME dataset** (*Aerial Imagery Pile burn detection using drones
(UAVs)*, IEEE Dataport) — aerial footage of controlled pile burns, which is about as
close to the real target as a public dataset gets.

It arrives as video (`.mp4`), not stills. That is the natural format for drone capture
but the wrong one for training an object detector, which wants individual labelled
frames.

Roboflow handles the conversion: upload the video, choose a frame-extraction rate, and
it splits the footage into discrete images. The rate matters more than it looks. Too
high and you get thousands of near-identical frames that inflate the dataset without
adding information — and worse, near-duplicates that straddle your train/validation
split will quietly inflate your validation scores. Too low and you lose the visual
variety that makes the model generalise.

## Step 2 — Labelling and dataset generation

Every frame is annotated with object detection bounding boxes using Roboflow's
annotation tool. This project uses a **single class: `fire`**, drawn around each
visible instance in the frame.

A one-class detector is a deliberate simplification. It sidesteps the genuinely hard
labelling question — where exactly does flame end and smoke begin — at the cost of not
distinguishing them at inference. For an early-warning system whose output is
"something is burning here, send someone", that tradeoff is the right one.

With labelling complete, the dataset is frozen into a **version**: an immutable
snapshot with its own train/validation/test split and preprocessing settings. Versions
are what make a training run reproducible — `version(1)` will hand back the same data
next year. This project trains against version 1 of the
`drone-fire-detection-byija` project.

## Step 3 — Training the model

With a labelled dataset version in hand, training is two moves: pull the dataset down
from Roboflow, then fine-tune a pretrained YOLO checkpoint on it.

Pulling the dataset:

```python
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tim-4ijf0").project("drone-fire-detection-byija")
dataset = project.version(1).download("yolov8")
```

And training:

```python
from ultralytics import YOLO

model = YOLO("yolo26m.pt")
train_results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=800,
    plots=True,
)
```

Two notes on that snippet, because both differ from the published article:

- **`download("yolov8")` names the dataset layout, not the model.** It means "images
  plus YOLO `.txt` labels plus a `data.yaml`" — the format YOLO26 consumes unchanged.
  The version number in that string has nothing to do with the version you train.
- **The original used `yolov8m.pt` via the `yolo task=detect mode=train` CLI.** This
  repo trains **YOLO26**, which is end-to-end and NMS-free, through the Python API.
  One practical consequence: there is no `iou` NMS threshold left to tune.

`imgsz=800` is on the large side, and deliberately so — fire in aerial footage is often
a small fraction of the frame, and downscaling to 640 throws away exactly the pixels
that matter. It costs memory; drop to 640, or to `yolo26n.pt`, if the GPU complains.

### What good looks like

The original YOLOv8 training run reported, on the `fire` class:

| Metric | Score |
|---|---|
| Recall | 0.95 |
| mAP@50 | 0.99 |
| mAP@50-95 | 0.63 |

Read those together rather than separately. Recall of 0.95 and mAP@50 of 0.99 say the
model reliably *finds* the fire — which is the thing that matters for an early-warning
system, where a miss is far more costly than a false alarm.

The drop to 0.63 at mAP@50-95 says the boxes are approximately, not precisely, placed.
That is expected and largely benign here: fire has no crisp boundary to agree on, so
tight-IoU scores are partly measuring the ambiguity of the label rather than a defect
in the model. Treat these as the benchmark to beat, not a guarantee — your own numbers
depend on your dataset version and split.

---

### ▶ Run notebook 1 of 3 — `drone_fire_detection_yolo26.ipynb`

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [View on GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb)

**Train the detector**

Downloads the dataset from Roboflow, fine-tunes YOLO26 for 50 epochs, validates the
result, runs predictions on the held-out test split, and exports the weights.

It ends by downloading **`best.pt`** — the trained checkpoint. Both remaining notebooks
need that file, and it is *not* committed to this repo, so keep it.

> **Before you start:** switch Colab to a GPU runtime (`Runtime` → `Change runtime type` → **T4 GPU**), and put your free [Roboflow API key](https://app.roboflow.com/settings/api) in Colab's 🔑 **Secrets** panel as `ROBOFLOW_API_KEY`.

---

## Step 4 — Deploying and testing

A trained checkpoint has a few possible destinations. You can push the weights back to
Roboflow and serve them through its hosted inference API, or run them on your own
hardware with [`roboflow/inference`](https://github.com/roboflow/inference) — the
sensible choice for a drone ground station, where round-tripping every frame to a
cloud API is neither fast nor reliable enough.

For evaluating whether the model actually works, though, the simplest path is to load
`best.pt` straight into a notebook and look at the output. That is what the next two
notebooks do, using **[Supervision](https://supervision.roboflow.com/)** to turn raw
model output into annotated images and video.

## Running inference on images

The image path is short. Read the image, run the model, convert the result into a
Supervision `Detections` object, and draw it:

```python
import cv2
import supervision as sv
from ultralytics import YOLO

model = YOLO("best.pt")
image = cv2.imread("fire_image.png")

result = model(image, conf=0.25, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)
```

`sv.Detections.from_ultralytics` is the useful join in that snippet. It normalises
YOLO's output into a structure the annotators consume, and carries across the class
names and — once we start tracking — the track IDs.

Then annotate. This is where the published code has genuinely broken rather than
merely aged:

```python
box_annotator = sv.BoxAnnotator(thickness=3)
label_annotator = sv.LabelAnnotator(text_scale=0.8)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections["class_name"], detections.confidence)
]

annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

sv.plot_image(image=cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), size=(10, 10))
```

The article passes `labels=` to `BoxAnnotator`. **That argument was removed in
supervision 0.22** — drawing is now split between `BoxAnnotator` (the box) and
`LabelAnnotator` (the text). Class names also now come from
`detections["class_name"]`, populated from the model itself, rather than being
looked up against a hand-written list that can drift out of sync with the weights.

---

### ▶ Run notebook 2 of 3 — `Supervision_Image_Inferencing.ipynb`

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Image_Inferencing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [View on GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Image_Inferencing.ipynb)

**Detect fire in a single image**

Loads `best.pt`, runs it over the sample `fire_image.png` from this repo, annotates
the detections with Supervision, and saves the result.

The quickest way to sanity-check a freshly trained checkpoint.

> **You will need `best.pt`** from notebook 1 — the notebook prompts you to upload it. A GPU is optional here; single-image inference is fine on CPU.

---

## Running inference on video

Video is where this gets interesting, because detection alone is not enough. Run a
detector frame by frame and you get a set of boxes per frame with no notion that the
fire in frame 200 is the same fire as in frame 199. There is no object permanence, so
you cannot count distinct fires, measure how long one has burned, or track whether it
is spreading.

**ByteTrack** supplies that continuity. Its central idea is worth understanding
because it drives how the code is written: most trackers throw away low-confidence
detections before matching. ByteTrack keeps them, and tries to match them against
already-established tracks. A fire briefly obscured by smoke or a banking aircraft
drops in confidence without ceasing to exist — and a weak box that lines up with a
confident track from the previous frame is almost certainly real.

### This is where the original code no longer runs at all

The published version installed ByteTrack like this:

```python
!git clone https://github.com/ifzhang/ByteTrack.git
!sed -i 's/onnx==1.8.1/onnx==1.9.0/g' requirements.txt
!pip3 install -q -r requirements.txt
!python3 setup.py -q develop
!pip install -q cython_bbox onemetric loguru lap thop
```

A source build of YOLOX, plus `onemetric` and `cython_bbox`, plus two `sed` patches
working around long-closed bugs. That toolchain no longer builds on current Python.

It is also no longer necessary. **ByteTrack ships inside Ultralytics now**, and the
entire block above collapses to one argument:

```python
result = model.track(
    frame,
    conf=0.1,
    persist=True,
    tracker="bytetrack.yaml",
    verbose=False,
)[0]
detections = sv.Detections.from_ultralytics(result)
detections = detections[detections.confidence >= 0.25]
```

`persist=True` is what carries tracker state between calls, so IDs stay stable across
frames. `from_ultralytics` reads those IDs into `detections.tracker_id`.

Note the two different confidence numbers, which is the ByteTrack insight turned into
code: **track at `0.1`, display at `0.25`**. Feeding the tracker only confident boxes
would discard exactly the weak detections it recovers tracks from. So track on
everything, then filter for drawing.

One footnote worth knowing if you compare against the article: its render loop
constructed a `BYTETracker` and then never called it inside the loop. The output video
was per-frame detections with no tracking applied.

---

### ▶ Run notebook 3 of 3 — `Supervision_Video_Inferencing.ipynb`

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Video_Inferencing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; [View on GitHub](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Video_Inferencing.ipynb)

**Detect and track fire across a video**

Runs YOLO26 + ByteTrack over `fire.mp4` frame by frame, draws boxes, labels, track
IDs and motion trails, and writes an annotated video.

It also re-encodes the output to H.264 with ffmpeg so it plays back inline — Supervision
writes `mp4v`, which browsers refuse to decode. That is why the result video appeared
to "not work" in the original.

> **You will need `best.pt`** from notebook 1, and a **T4 GPU** runtime — video inference on CPU is slow enough to be impractical.

---

## Optional: check your environment

Nothing above needs executing — it is a map, not a pipeline. But if you want to
confirm this runtime is set up before jumping into notebook 1, run the cell below.

In [ ]:
import shutil
import subprocess
import sys

print(f"Python {sys.version.split()[0]}\n")

gpu = shutil.which("nvidia-smi")
if gpu:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f"GPU: {out or 'detected'}")
else:
    print("GPU: none detected — set Runtime > Change runtime type > T4 GPU")

print("ffmpeg:", "yes" if shutil.which("ffmpeg") else "no (needed by notebook 3)")

for pkg in ("ultralytics", "supervision", "roboflow"):
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'installed')}")
    except ImportError:
        print(f"{pkg}: not installed (each notebook installs its own)")

## What changed since the article was published

The write-up dates from September 2023. Its code targets libraries that have since
moved on — in a few places far enough that it no longer runs. Everything in this repo
has been updated:

| | Article (2023) | This repo |
|---|---|---|
| Model | `yolov8m.pt` | **`yolo26m.pt`** — end-to-end, NMS-free |
| Ultralytics | `==8.0.20` | `>=8.4.122` |
| Supervision | `==0.1.0` | `>=0.30.0` |
| Roboflow | unpinned | `>=1.4.1` |
| Training | `yolo task=detect mode=train` CLI | Python API, `results.save_dir` |
| Annotation | `BoxAnnotator(labels=...)` | `BoxAnnotator` + `LabelAnnotator` |
| Tracking | ByteTrack git clone + YOLOX source build | Built into Ultralytics |
| API key | `api_key="YOUR_API_KEY"` in a cell | Colab Secrets |
| Video output | `mp4v` (browsers cannot decode) | H.264 re-encode |

The notebooks also would not open. Two separate faults: the video notebook carried a
`metadata.widgets` block missing its required `state` key — the exact trigger for
GitHub's *"Invalid Notebook"* error — and the training notebook was 1.68 MB of embedded
base64 outputs, over GitHub's ~1 MB render ceiling. All notebooks are now `nbformat 4.5`
with cell IDs and outputs stripped.

## Conclusion

The system is a custom-trained YOLO26 detector that finds fire in aerial imagery, wired
to a tracker that follows each fire across video frames. Around it sits the operational
loop that makes it useful: a drone covering ground nobody could patrol on foot, and a
control centre that turns a detection into a dispatched crew.

None of the individual pieces are exotic. What makes the approach work is that the
expensive part — sustained visual attention across large, remote terrain — is exactly
what a model is good at and a human is not.

### Where to go next

1. **[`drone_fire_detection_yolo26.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb)** — train the model, export `best.pt`
2. **[`Supervision_Image_Inferencing.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Image_Inferencing.ipynb)** — test it on an image
3. **[`Supervision_Video_Inferencing.ipynb`](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/Supervision_Video_Inferencing.ipynb)** — test it on video, with tracking

Worth exploring beyond that: deploying to a ground station with
[`roboflow/inference`](https://github.com/roboflow/inference) rather than a cloud API,
splitting `fire` into separate flame and smoke classes (smoke is visible from
considerably further away), and using the track IDs to measure spread rate rather than
just presence.

### Sources and credit

- Original write-up: [Aerial Fire Detection with Drone Imagery and Computer Vision](https://blog.roboflow.com/aerial-fire-detection/) — Timothy M., Roboflow Blog
- Upstream repository: [tim3in/Fire-Detection-Drone](https://github.com/tim3in/Fire-Detection-Drone)
- Dataset: [`drone-fire-detection-byija`](https://universe.roboflow.com/tim-4ijf0/drone-fire-detection-byija) on Roboflow Universe

**Dataset citation**

> Alireza Shamsoshoara, Fatemeh Afghah, Abolfazl Razi, Liming Zheng, Peter Fulé, Erik
> Blasch, November 19, 2020, "The FLAME dataset: Aerial Imagery Pile burn detection
> using drones (UAVs)", IEEE Dataport, doi: https://dx.doi.org/10.21227/qad6-r683

**Article citation**

> Timothy M. (Sep 19, 2023). Aerial Fire Detection with Drone Imagery and Computer
> Vision. Roboflow Blog: https://blog.roboflow.com/aerial-fire-detection/